# Order and chain polytopes — full exploration, four example posets

Stanley's order polytope \(O(P)\) and chain polytope \(C(P)\) (R. Stanley,
*Two poset polytopes*, Discrete Comput. Geom. 1 (1986) 9-23) attach a pair
of \((0,1)\)-polytopes to any finite poset \(P\) on \(n\) elements. Both live
in \(\mathbb{R}^n\), both have dimension \(n\), and both have volume
\(e(P)/n!\), where \(e(P)\) is the number of linear extensions of \(P\) — but
in general they are *not* combinatorially equivalent. This notebook works
through four posets chosen to span that range, matching
`order_chain_polytopes.sage` and the site's own
[order-chain-polytopes.md](../docs/catalog/order-chain-polytopes.md):

1. **the antichain on 3 elements** — no relations at all, so \(O(P)\) and
   \(C(P))\) turn out to be literally the same polytope, the 3-cube.
2. **the chain on 4 elements** — totally ordered; \(O(P)\) and \(C(P)\) are
   different embeddings of the same combinatorial type, a 4-simplex.
3. **the "N" (fence) poset on 4 elements** — genuinely nontrivial (neither a
   chain nor an antichain); here \(O(P)\) and \(C(P)\) happen to share the
   same f-vector even though their vertex coordinates differ.
4. **a graded rank-\((2,2,2)\) poset on 6 elements** ("double diamond":
   complete bipartite between consecutive ranks) — found by an exhaustive
   search over `Posets(6)` while building this family — where \(O(P)\) and
   \(C(P)\) have genuinely *different* f-vectors beyond \(f_0\). This is the
   general case, not an exception; it also lets us check the open
   **Hibi–Li conjecture** (that \(C(P)\)'s f-vector dominates \(O(P)\)'s,
   entry by entry) on a concrete example.

For each polytope, this notebook:

1. lists all vertices (`order_polytope_vertices`/`chain_polytope_vertices`
   in `common.sage`, which delegate to Sage's own `Poset.order_polytope()`/
   `.chain_polytope()`),
2. computes the canonical form via the general nbc method (Brown–Dupont
   Prop. 6.7) — these polytopes aren't simple in general, so Prop. 6.10
   doesn't apply uniformly the way it does for the graph-associahedra
   family,
3. computes the projective (polar) dual,
4. checks the **volume conjecture** at the centroid,
5. enumerates all triangulations and identifies which are regular,
6. computes the secondary polytope and its vertex embedding.

**No steps are skipped anywhere in this notebook** — all four posets, both
polytopes each, stayed well under a minute for every step (measured while
building this), unlike most of the other families in this catalog. The
"double diamond" \(O(P)\)/\(C(P)\) is the most expensive pair here (\(d=6\),
10 vertices) and still finishes its full six-step treatment in well under
a minute.

**Requires the `sagemath` Jupyter kernel** and must be opened from the same
synced folder as the `.sage` files — see `README.md`.

In [ ]:
load("general_canonical_forms.sage")

That `load` pulls in `common.sage` (poset-polytope vertex generators,
`polar_dual`, `secondary_polytope_data`) and `vertex_sum_canonical_forms.sage`
too, and runs `general_canonical_forms.sage`'s own test suite as a side
effect (scroll up for that PASS/FAIL output). Everything below is fresh,
per-poset exploration specific to this family.

## 1. The antichain on 3 elements — O(P) = C(P), literally the 3-cube

In [ ]:
antichain3 = Poset({1: [], 2: [], 3: []})
print("elements:", antichain3.list())
print("cover relations:", antichain3.cover_relations())
print("linear extensions:", len(list(antichain3.linear_extensions())), "(all 3! orderings, since nothing is comparable)")

### O(antichain_3)

In [ ]:
pts = order_polytope_vertices(antichain3)
d = len(pts[0])
y = [var(f"y{i}") for i in range(1, d + 1)]
P = Polyhedron(vertices=pts)
print(f"{len(pts)} vertices, dimension {P.dimension()}:")
pts

#### Canonical form (Proposition 6.7, general nbc method)

In [ ]:
phi = general_canonical_form_density(P, y)
verify_pole_structure(f"O(antichain_3)", phi, P, y)
phi

#### Canonical form, broken down by vertex

In [ ]:
rows = canonical_form_by_vertex(P, y)
print_canonical_form_by_vertex(rows)

#### Projective dual

In [ ]:
Dual = polar_dual(P)
print(f"dual: {Dual.n_vertices()} vertices, {Dual.n_facets()} facets")
Dual.vertices_list()

#### Volume conjecture: canonical form vs. the volume of the projective dual (at the centroid)

In [ ]:
centroid = [sum(QQ(v[i]) for v in pts) / len(pts) for i in range(d)]
pts_centered = [tuple(QQ(v[i]) - centroid[i] for i in range(d)) for v in pts]
P_centered = Polyhedron(vertices=pts_centered)

phi_centroid = general_canonical_form_density(P_centered, y)
val_at_centroid = phi_centroid.subs({yi: 0 for yi in y})

vol_dual = Dual.volume()
target = factorial(d) * vol_dual
print("phi at the centroid =", val_at_centroid)
print("d! * Vol(dual) =", target)
match_plus = bool((val_at_centroid - target) == 0)
match_minus = bool((val_at_centroid + target) == 0)
print("matches d! * Vol(dual):", match_plus, " matches -d! * Vol(dual):", match_minus)
assert match_plus or match_minus, "volume-conjecture identity failed -- would be a real bug"

#### All triangulations, and which are regular

In [ ]:
sp, sp_reduced, rows = secondary_polytope_data(pts)
for t, gkz, is_reg in rows:
    print(t, "GKZ vector:", gkz, " regular:", is_reg)
print(f"{len(rows)} triangulation(s) total, {sum(1 for _, _, r in rows if r)} regular")

#### Secondary polytope: vertex embedding

In [ ]:
print(f"secondary polytope: dimension {sp_reduced.dimension()}, {sp_reduced.n_vertices()} vertex/vertices")
sp_reduced.vertices_list()

### C(antichain_3)

In [ ]:
pts = chain_polytope_vertices(antichain3)
d = len(pts[0])
y = [var(f"y{i}") for i in range(1, d + 1)]
P = Polyhedron(vertices=pts)
print(f"{len(pts)} vertices, dimension {P.dimension()}:")
pts

#### Canonical form (Proposition 6.7, general nbc method)

In [ ]:
phi = general_canonical_form_density(P, y)
verify_pole_structure(f"C(antichain_3)", phi, P, y)
phi

#### Canonical form, broken down by vertex

In [ ]:
rows = canonical_form_by_vertex(P, y)
print_canonical_form_by_vertex(rows)

#### Projective dual

In [ ]:
Dual = polar_dual(P)
print(f"dual: {Dual.n_vertices()} vertices, {Dual.n_facets()} facets")
Dual.vertices_list()

#### Volume conjecture: canonical form vs. the volume of the projective dual (at the centroid)

In [ ]:
centroid = [sum(QQ(v[i]) for v in pts) / len(pts) for i in range(d)]
pts_centered = [tuple(QQ(v[i]) - centroid[i] for i in range(d)) for v in pts]
P_centered = Polyhedron(vertices=pts_centered)

phi_centroid = general_canonical_form_density(P_centered, y)
val_at_centroid = phi_centroid.subs({yi: 0 for yi in y})

vol_dual = Dual.volume()
target = factorial(d) * vol_dual
print("phi at the centroid =", val_at_centroid)
print("d! * Vol(dual) =", target)
match_plus = bool((val_at_centroid - target) == 0)
match_minus = bool((val_at_centroid + target) == 0)
print("matches d! * Vol(dual):", match_plus, " matches -d! * Vol(dual):", match_minus)
assert match_plus or match_minus, "volume-conjecture identity failed -- would be a real bug"

#### All triangulations, and which are regular

In [ ]:
sp, sp_reduced, rows = secondary_polytope_data(pts)
for t, gkz, is_reg in rows:
    print(t, "GKZ vector:", gkz, " regular:", is_reg)
print(f"{len(rows)} triangulation(s) total, {sum(1 for _, _, r in rows if r)} regular")

#### Secondary polytope: vertex embedding

In [ ]:
print(f"secondary polytope: dimension {sp_reduced.dimension()}, {sp_reduced.n_vertices()} vertex/vertices")
sp_reduced.vertices_list()

With no relations at all, *every* subset of \(\{1,2,3\}\) is simultaneously
an order filter (there's nothing to violate up-closure) and an antichain
(nothing is comparable) — so \(O(P)\) and \(C(P)\) are built from exactly the
same \(2^3=8\) indicator vectors: the full cube \(\{0,1\}^3\), not just two
polytopes that happen to share a combinatorial type. Confirmed above: the
two vertex lists are the same 8 points, just possibly listed in a
different order.

## 2. The chain on 4 elements — O(P) and C(P), two different embeddings of the same 4-simplex

In [ ]:
chain4 = Poset({1: [2], 2: [3], 3: [4]})
print("elements:", chain4.list())
print("cover relations:", chain4.cover_relations())
print("linear extensions:", len(list(chain4.linear_extensions())), "(only one: the poset is already totally ordered)")

### O(chain_4)

In [ ]:
pts = order_polytope_vertices(chain4)
d = len(pts[0])
y = [var(f"y{i}") for i in range(1, d + 1)]
P = Polyhedron(vertices=pts)
print(f"{len(pts)} vertices, dimension {P.dimension()}:")
pts

#### Canonical form (Proposition 6.7, general nbc method)

In [ ]:
phi = general_canonical_form_density(P, y)
verify_pole_structure(f"O(chain_4)", phi, P, y)
phi

#### Canonical form, broken down by vertex

In [ ]:
rows = canonical_form_by_vertex(P, y)
print_canonical_form_by_vertex(rows)

#### Projective dual

In [ ]:
Dual = polar_dual(P)
print(f"dual: {Dual.n_vertices()} vertices, {Dual.n_facets()} facets")
Dual.vertices_list()

#### Volume conjecture: canonical form vs. the volume of the projective dual (at the centroid)

In [ ]:
centroid = [sum(QQ(v[i]) for v in pts) / len(pts) for i in range(d)]
pts_centered = [tuple(QQ(v[i]) - centroid[i] for i in range(d)) for v in pts]
P_centered = Polyhedron(vertices=pts_centered)

phi_centroid = general_canonical_form_density(P_centered, y)
val_at_centroid = phi_centroid.subs({yi: 0 for yi in y})

vol_dual = Dual.volume()
target = factorial(d) * vol_dual
print("phi at the centroid =", val_at_centroid)
print("d! * Vol(dual) =", target)
match_plus = bool((val_at_centroid - target) == 0)
match_minus = bool((val_at_centroid + target) == 0)
print("matches d! * Vol(dual):", match_plus, " matches -d! * Vol(dual):", match_minus)
assert match_plus or match_minus, "volume-conjecture identity failed -- would be a real bug"

#### All triangulations, and which are regular

In [ ]:
sp, sp_reduced, rows = secondary_polytope_data(pts)
for t, gkz, is_reg in rows:
    print(t, "GKZ vector:", gkz, " regular:", is_reg)
print(f"{len(rows)} triangulation(s) total, {sum(1 for _, _, r in rows if r)} regular")

#### Secondary polytope: vertex embedding

In [ ]:
print(f"secondary polytope: dimension {sp_reduced.dimension()}, {sp_reduced.n_vertices()} vertex/vertices")
sp_reduced.vertices_list()

### C(chain_4)

In [ ]:
pts = chain_polytope_vertices(chain4)
d = len(pts[0])
y = [var(f"y{i}") for i in range(1, d + 1)]
P = Polyhedron(vertices=pts)
print(f"{len(pts)} vertices, dimension {P.dimension()}:")
pts

#### Canonical form (Proposition 6.7, general nbc method)

In [ ]:
phi = general_canonical_form_density(P, y)
verify_pole_structure(f"C(chain_4)", phi, P, y)
phi

#### Canonical form, broken down by vertex

In [ ]:
rows = canonical_form_by_vertex(P, y)
print_canonical_form_by_vertex(rows)

#### Projective dual

In [ ]:
Dual = polar_dual(P)
print(f"dual: {Dual.n_vertices()} vertices, {Dual.n_facets()} facets")
Dual.vertices_list()

#### Volume conjecture: canonical form vs. the volume of the projective dual (at the centroid)

In [ ]:
centroid = [sum(QQ(v[i]) for v in pts) / len(pts) for i in range(d)]
pts_centered = [tuple(QQ(v[i]) - centroid[i] for i in range(d)) for v in pts]
P_centered = Polyhedron(vertices=pts_centered)

phi_centroid = general_canonical_form_density(P_centered, y)
val_at_centroid = phi_centroid.subs({yi: 0 for yi in y})

vol_dual = Dual.volume()
target = factorial(d) * vol_dual
print("phi at the centroid =", val_at_centroid)
print("d! * Vol(dual) =", target)
match_plus = bool((val_at_centroid - target) == 0)
match_minus = bool((val_at_centroid + target) == 0)
print("matches d! * Vol(dual):", match_plus, " matches -d! * Vol(dual):", match_minus)
assert match_plus or match_minus, "volume-conjecture identity failed -- would be a real bug"

#### All triangulations, and which are regular

In [ ]:
sp, sp_reduced, rows = secondary_polytope_data(pts)
for t, gkz, is_reg in rows:
    print(t, "GKZ vector:", gkz, " regular:", is_reg)
print(f"{len(rows)} triangulation(s) total, {sum(1 for _, _, r in rows if r)} regular")

#### Secondary polytope: vertex embedding

In [ ]:
print(f"secondary polytope: dimension {sp_reduced.dimension()}, {sp_reduced.n_vertices()} vertex/vertices")
sp_reduced.vertices_list()

For a totally ordered \(P\), the order filters are exactly the \(n+1\)
"suffixes" (including the empty one), nested inside each other, and the
antichains are exactly the \(n+1\) singletons plus the empty set — two
different-looking but affinely independent sets of \(0/1\) points, each
forming an \(n\)-simplex. Both have volume \(1/4! = e(P)/4!\) with
\(e(P)=1\), matching `order_chain_polytopes.sage`.

## 3. The "N" (fence) poset on 4 elements — same f-vector, different embeddings

In [ ]:
N = Poset({"a": ["c"], "b": ["c", "d"]})
print("elements:", N.list())
print("cover relations:", N.cover_relations())
print("linear extensions:", len(list(N.linear_extensions())))

### O(N)

In [ ]:
pts = order_polytope_vertices(N)
d = len(pts[0])
y = [var(f"y{i}") for i in range(1, d + 1)]
P = Polyhedron(vertices=pts)
print(f"{len(pts)} vertices, dimension {P.dimension()}:")
pts

#### Canonical form (Proposition 6.7, general nbc method)

In [ ]:
phi = general_canonical_form_density(P, y)
verify_pole_structure(f"O(N)", phi, P, y)
phi

#### Canonical form, broken down by vertex

In [ ]:
rows = canonical_form_by_vertex(P, y)
print_canonical_form_by_vertex(rows)

#### Projective dual

In [ ]:
Dual = polar_dual(P)
print(f"dual: {Dual.n_vertices()} vertices, {Dual.n_facets()} facets")
Dual.vertices_list()

#### Volume conjecture: canonical form vs. the volume of the projective dual (at the centroid)

In [ ]:
centroid = [sum(QQ(v[i]) for v in pts) / len(pts) for i in range(d)]
pts_centered = [tuple(QQ(v[i]) - centroid[i] for i in range(d)) for v in pts]
P_centered = Polyhedron(vertices=pts_centered)

phi_centroid = general_canonical_form_density(P_centered, y)
val_at_centroid = phi_centroid.subs({yi: 0 for yi in y})

vol_dual = Dual.volume()
target = factorial(d) * vol_dual
print("phi at the centroid =", val_at_centroid)
print("d! * Vol(dual) =", target)
match_plus = bool((val_at_centroid - target) == 0)
match_minus = bool((val_at_centroid + target) == 0)
print("matches d! * Vol(dual):", match_plus, " matches -d! * Vol(dual):", match_minus)
assert match_plus or match_minus, "volume-conjecture identity failed -- would be a real bug"

#### All triangulations, and which are regular

In [ ]:
sp, sp_reduced, rows = secondary_polytope_data(pts)
for t, gkz, is_reg in rows:
    print(t, "GKZ vector:", gkz, " regular:", is_reg)
print(f"{len(rows)} triangulation(s) total, {sum(1 for _, _, r in rows if r)} regular")

#### Secondary polytope: vertex embedding

In [ ]:
print(f"secondary polytope: dimension {sp_reduced.dimension()}, {sp_reduced.n_vertices()} vertex/vertices")
sp_reduced.vertices_list()

### C(N)

In [ ]:
pts = chain_polytope_vertices(N)
d = len(pts[0])
y = [var(f"y{i}") for i in range(1, d + 1)]
P = Polyhedron(vertices=pts)
print(f"{len(pts)} vertices, dimension {P.dimension()}:")
pts

#### Canonical form (Proposition 6.7, general nbc method)

In [ ]:
phi = general_canonical_form_density(P, y)
verify_pole_structure(f"C(N)", phi, P, y)
phi

#### Canonical form, broken down by vertex

In [ ]:
rows = canonical_form_by_vertex(P, y)
print_canonical_form_by_vertex(rows)

#### Projective dual

In [ ]:
Dual = polar_dual(P)
print(f"dual: {Dual.n_vertices()} vertices, {Dual.n_facets()} facets")
Dual.vertices_list()

#### Volume conjecture: canonical form vs. the volume of the projective dual (at the centroid)

In [ ]:
centroid = [sum(QQ(v[i]) for v in pts) / len(pts) for i in range(d)]
pts_centered = [tuple(QQ(v[i]) - centroid[i] for i in range(d)) for v in pts]
P_centered = Polyhedron(vertices=pts_centered)

phi_centroid = general_canonical_form_density(P_centered, y)
val_at_centroid = phi_centroid.subs({yi: 0 for yi in y})

vol_dual = Dual.volume()
target = factorial(d) * vol_dual
print("phi at the centroid =", val_at_centroid)
print("d! * Vol(dual) =", target)
match_plus = bool((val_at_centroid - target) == 0)
match_minus = bool((val_at_centroid + target) == 0)
print("matches d! * Vol(dual):", match_plus, " matches -d! * Vol(dual):", match_minus)
assert match_plus or match_minus, "volume-conjecture identity failed -- would be a real bug"

#### All triangulations, and which are regular

In [ ]:
sp, sp_reduced, rows = secondary_polytope_data(pts)
for t, gkz, is_reg in rows:
    print(t, "GKZ vector:", gkz, " regular:", is_reg)
print(f"{len(rows)} triangulation(s) total, {sum(1 for _, _, r in rows if r)} regular")

#### Secondary polytope: vertex embedding

In [ ]:
print(f"secondary polytope: dimension {sp_reduced.dimension()}, {sp_reduced.n_vertices()} vertex/vertices")
sp_reduced.vertices_list()

\(N\) is neither a chain nor an antichain (\(a\) and \(b\) are incomparable,
as are \(c\) and \(d\)), yet \(O(N)\) and \(C(N)\) turn out to share the same
f-vector \((8,18,17,7)\) — checked above and in `order_chain_polytopes.sage`.
That's not guaranteed by Stanley's theorem (which only guarantees equal
*volume*) — it's a coincidence of this particular small poset, and the
next section shows a poset where it fails.

## 4. A rank-(2,2,2) poset on 6 elements — the general case: O(P) and C(P) genuinely differ

In [ ]:
double_diamond = Poset({0: [2, 3], 1: [2, 3], 2: [4, 5], 3: [4, 5]})
print("elements:", double_diamond.list())
print("cover relations:", double_diamond.cover_relations())
print("linear extensions:", len(list(double_diamond.linear_extensions())))

### O(double_diamond)

In [ ]:
pts = order_polytope_vertices(double_diamond)
d = len(pts[0])
y = [var(f"y{i}") for i in range(1, d + 1)]
P = Polyhedron(vertices=pts)
print(f"{len(pts)} vertices, dimension {P.dimension()}:")
pts

#### Canonical form (Proposition 6.7, general nbc method)

In [ ]:
phi = general_canonical_form_density(P, y)
verify_pole_structure(f"O(double_diamond)", phi, P, y)
phi

#### Canonical form, broken down by vertex

In [ ]:
rows = canonical_form_by_vertex(P, y)
print_canonical_form_by_vertex(rows)

#### Projective dual

In [ ]:
Dual = polar_dual(P)
print(f"dual: {Dual.n_vertices()} vertices, {Dual.n_facets()} facets")
Dual.vertices_list()

#### Volume conjecture: canonical form vs. the volume of the projective dual (at the centroid)

In [ ]:
centroid = [sum(QQ(v[i]) for v in pts) / len(pts) for i in range(d)]
pts_centered = [tuple(QQ(v[i]) - centroid[i] for i in range(d)) for v in pts]
P_centered = Polyhedron(vertices=pts_centered)

phi_centroid = general_canonical_form_density(P_centered, y)
val_at_centroid = phi_centroid.subs({yi: 0 for yi in y})

vol_dual = Dual.volume()
target = factorial(d) * vol_dual
print("phi at the centroid =", val_at_centroid)
print("d! * Vol(dual) =", target)
match_plus = bool((val_at_centroid - target) == 0)
match_minus = bool((val_at_centroid + target) == 0)
print("matches d! * Vol(dual):", match_plus, " matches -d! * Vol(dual):", match_minus)
assert match_plus or match_minus, "volume-conjecture identity failed -- would be a real bug"

#### All triangulations, and which are regular

In [ ]:
sp, sp_reduced, rows = secondary_polytope_data(pts)
for t, gkz, is_reg in rows:
    print(t, "GKZ vector:", gkz, " regular:", is_reg)
print(f"{len(rows)} triangulation(s) total, {sum(1 for _, _, r in rows if r)} regular")

#### Secondary polytope: vertex embedding

In [ ]:
print(f"secondary polytope: dimension {sp_reduced.dimension()}, {sp_reduced.n_vertices()} vertex/vertices")
sp_reduced.vertices_list()

### C(double_diamond)

In [ ]:
pts = chain_polytope_vertices(double_diamond)
d = len(pts[0])
y = [var(f"y{i}") for i in range(1, d + 1)]
P = Polyhedron(vertices=pts)
print(f"{len(pts)} vertices, dimension {P.dimension()}:")
pts

#### Canonical form (Proposition 6.7, general nbc method)

In [ ]:
phi = general_canonical_form_density(P, y)
verify_pole_structure(f"C(double_diamond)", phi, P, y)
phi

#### Canonical form, broken down by vertex

In [ ]:
rows = canonical_form_by_vertex(P, y)
print_canonical_form_by_vertex(rows)

#### Projective dual

In [ ]:
Dual = polar_dual(P)
print(f"dual: {Dual.n_vertices()} vertices, {Dual.n_facets()} facets")
Dual.vertices_list()

#### Volume conjecture: canonical form vs. the volume of the projective dual (at the centroid)

In [ ]:
centroid = [sum(QQ(v[i]) for v in pts) / len(pts) for i in range(d)]
pts_centered = [tuple(QQ(v[i]) - centroid[i] for i in range(d)) for v in pts]
P_centered = Polyhedron(vertices=pts_centered)

phi_centroid = general_canonical_form_density(P_centered, y)
val_at_centroid = phi_centroid.subs({yi: 0 for yi in y})

vol_dual = Dual.volume()
target = factorial(d) * vol_dual
print("phi at the centroid =", val_at_centroid)
print("d! * Vol(dual) =", target)
match_plus = bool((val_at_centroid - target) == 0)
match_minus = bool((val_at_centroid + target) == 0)
print("matches d! * Vol(dual):", match_plus, " matches -d! * Vol(dual):", match_minus)
assert match_plus or match_minus, "volume-conjecture identity failed -- would be a real bug"

#### All triangulations, and which are regular

In [ ]:
sp, sp_reduced, rows = secondary_polytope_data(pts)
for t, gkz, is_reg in rows:
    print(t, "GKZ vector:", gkz, " regular:", is_reg)
print(f"{len(rows)} triangulation(s) total, {sum(1 for _, _, r in rows if r)} regular")

#### Secondary polytope: vertex embedding

In [ ]:
print(f"secondary polytope: dimension {sp_reduced.dimension()}, {sp_reduced.n_vertices()} vertex/vertices")
sp_reduced.vertices_list()

### f-vectors side by side, and the Hibi–Li dominance check

In [ ]:
fO = Polyhedron(vertices=order_polytope_vertices(double_diamond)).f_vector()
fC = Polyhedron(vertices=chain_polytope_vertices(double_diamond)).f_vector()
print("f(O(double_diamond)) =", tuple(fO))
print("f(C(double_diamond)) =", tuple(fC))
dominates = all(c >= o for o, c in zip(fO, fC))
print("C(P) dominates O(P) entrywise:", dominates, "(Hibi-Li conjecture, open in general -- consistent here, not proved)")
assert fO != fC, "expected these to differ -- that's the whole point of this example"
assert dominates

\(O(P)\) and \(C(P)\) have the same number of vertices (\(f_0=10\) both —
Stanley's theorem does guarantee that much: vertices of both correspond to
antichains of \(P\)) but diverge from \(f_1\) onward: \((39,77,82,46,12)\) vs.
\((39,78,86,51,14)\). They still have exactly the same volume, \(1/90\) (both
above, and in `order_chain_polytopes.sage`) — Stanley's theorem is about
volume/Ehrhart equivalence, not combinatorial equivalence, and this is the
example that shows the difference concretely rather than just asserting it.
The Hibi–Li conjecture (Hibi, N. Li) — that \(C(P)\)'s f-vector always
dominates \(O(P)\)'s, entry by entry — is open in general; it holds here.